# 📖 Notebook 10: Temporal UI & CLI — Managing Workflows in Production

In production, you need tools to inspect, debug, and manage workflows. Writing workflow code is only half the job; the other half is operating live executions safely when something is stuck, slow, or waiting for a human decision.

Temporal gives you two operator-friendly surfaces: the **Web UI** and the **`temporal` CLI**. This notebook shows how to use both against real running workflows.

## Learning Objectives

- Navigate the Temporal Web UI
- Use the `temporal` CLI for workflow management
- List, describe, and terminate workflows
- Send signals and queries via CLI
- Manage namespaces
- Search workflows using visibility queries


## 🛠️ Setup

Start the local stack and install Python dependencies:

```bash
cd 03-technologies/workflow-engines/temporal
docker compose up -d
uv sync
```

If the `temporal` CLI is not installed yet, install it once with Homebrew:

```bash
brew install temporal
```

- Temporal Web UI: http://localhost:8080
- Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
- If the kernel does not appear, reload the VS Code window: `Cmd+Shift+P` → `Reload Window`.
- In notebook code cells, a leading `!` means "run this shell command".


In [ ]:
import asyncio

from temporalio import workflow
from temporalio.client import Client
from temporalio.service import RPCError
from temporalio.worker import Worker

client = await Client.connect("localhost:7233")
print("✅ Connected to Temporal on localhost:7233")


## A Long-Running Workflow We Can Inspect

The easiest way to learn the UI and CLI is to create workflows that stay alive long enough for us to inspect them. This workflow keeps a little bit of in-memory state, exposes a **query** to read that state, and listens for **signals** to change it.

```
signal process_item ──▶ workflow state ──▶ query get_status
signal shutdown     ──▶ workflow exits ──▶ final result
```


In [ ]:
@workflow.defn
class LongRunningWorkflow:
    def __init__(self):
        self._status = "started"
        self._items_processed = 0
        self._shutdown = False

    @workflow.signal
    def process_item(self, item: str):
        self._items_processed += 1
        self._status = f"processed {self._items_processed} items"

    @workflow.signal
    def shutdown(self):
        self._shutdown = True

    @workflow.query
    def get_status(self) -> str:
        return self._status

    @workflow.run
    async def run(self) -> str:
        self._status = "running"
        await workflow.wait_condition(lambda: self._shutdown)
        return f"Completed after processing {self._items_processed} items"


print("✅ Defined LongRunningWorkflow with signals and a query")


## Start Several Workflows

We will start five workflows with stable IDs so the UI and CLI commands later in the notebook can reference them directly. We also keep a worker running in the background so signals and queries have an active poller to talk to.

In [ ]:
TASK_QUEUE = "ui-cli-demo-queue"
WORKFLOW_IDS = [f"demo-workflow-{i}" for i in range(5)]

if "worker_task" in globals() and not worker_task.done():
    worker_task.cancel()
    try:
        await worker_task
    except asyncio.CancelledError:
        pass

for workflow_id in WORKFLOW_IDS:
    handle = client.get_workflow_handle(workflow_id)
    try:
        await handle.terminate("Reset notebook state")
    except RPCError:
        pass

worker = Worker(
    client,
    task_queue=TASK_QUEUE,
    workflows=[LongRunningWorkflow],
)
worker_task = asyncio.create_task(worker.run())
await asyncio.sleep(1)

handles = []
for workflow_id in WORKFLOW_IDS:
    handle = await client.start_workflow(
        LongRunningWorkflow.run,
        id=workflow_id,
        task_queue=TASK_QUEUE,
    )
    handles.append(handle)

print("✅ Started workflows:")
for workflow_id in WORKFLOW_IDS:
    print(f"   - {workflow_id}")
print("Refresh the Web UI list view to see them.")


## Temporal Web UI Walkthrough

Open http://localhost:8080 and look for these areas:

- **Namespace selector** — choose the namespace you want to inspect. For local labs, this is usually `default`.
- **Workflow list view** — shows running, completed, failed, and terminated workflows. You should see `demo-workflow-0` through `demo-workflow-4`.
- **Workflow detail view** — click one workflow to inspect the event history, timeline, pending tasks, and failure details if something breaks.
- **Input / Output tab** — useful for checking workflow arguments and final result payloads.
- **Queries tab** — lets you run queries against live workflows, which is great for support and debugging.

A nice mental model is: **the UI is your dashboard, the CLI is your scalpel**.

## Temporal CLI Basics

The `temporal` CLI talks to the same cluster as the UI, but it is scriptable and automation-friendly. In a notebook, prefix commands with `!`. In a terminal, run the same command without the `!`.

In [ ]:
# List running workflows
!temporal workflow list --namespace default

# Describe a specific workflow
!temporal workflow describe --workflow-id demo-workflow-0


The first command shows a table of workflow executions in the namespace. The second command drills into one execution and prints identifiers, status, task queue, run ID, start time, and other metadata that helps you answer "what is this workflow doing right now?"

In [ ]:
# Query a workflow
!temporal workflow query --workflow-id demo-workflow-0 --type get_status

# Signal a workflow
!temporal workflow signal --workflow-id demo-workflow-0 --name process_item --input '"item-A"'

# Query again to see the updated status
!temporal workflow query --workflow-id demo-workflow-0 --type get_status


The first query should return something like `running`. After the signal, the workflow updates its in-memory state, so the second query should show `processed 1 items`. This is a great example of the difference between **queries** (read state) and **signals** (change state).

In [ ]:
# Terminate a workflow
!temporal workflow terminate --workflow-id demo-workflow-4 --reason "Demo cleanup"


Termination is the emergency stop button. Use it when you are sure the workflow should end immediately, even if it never gets a chance to run cleanup logic inside the workflow code. In production, prefer **signals** when you want a graceful shutdown and **termination** when you need a hard stop.

## Visibility Queries

Visibility queries let you search by workflow metadata instead of a single ID. This is the CLI equivalent of filtering a dashboard. The first command filters by workflow type, the second filters by status, and the third combines both conditions so you can narrow the list to exactly the executions you care about.

In [ ]:
# Search for workflows by type
!temporal workflow list --query 'WorkflowType = "LongRunningWorkflow"'

# Search by status
!temporal workflow list --query 'ExecutionStatus = "Running"'

# Combine filters
!temporal workflow list --query 'WorkflowType = "LongRunningWorkflow" AND ExecutionStatus = "Running"'


## Namespace Management

Namespaces are like logical tenants or environments inside Temporal. Many teams use them to separate dev, staging, and production, or to isolate different business domains. `namespace list` shows what namespaces exist, and `namespace describe` prints retention, archival, and replication settings for one namespace.

In [ ]:
# List namespaces
!temporal operator namespace list

# Describe namespace
!temporal operator namespace describe --namespace default


## Batch Operations

One of the biggest reasons to learn the CLI is batch management. If a bad deployment starts hundreds of workflows with the wrong parameters, you do not want to terminate them one by one. The next command applies one action to every workflow that matches the query, which is much closer to how real production cleanup works.

In [ ]:
# Terminate all matching workflows
!temporal workflow terminate --query 'WorkflowType = "LongRunningWorkflow" AND ExecutionStatus = "Running"' --reason "Batch cleanup"


## Clean Up

The batch terminate command should stop most live runs. This final cell is intentionally idempotent: it asks the Python client to terminate any survivors and then shuts down the background worker task.

In [ ]:
for workflow_id in WORKFLOW_IDS:
    handle = client.get_workflow_handle(workflow_id)
    try:
        await handle.terminate("Notebook cleanup")
    except RPCError:
        pass

worker_task.cancel()
try:
    await worker_task
except asyncio.CancelledError:
    pass

print("✅ Cleaned up workflows and stopped the background worker")


## CLI Cheat Sheet

| Task | CLI Command |
|---|---|
| List workflows | `temporal workflow list` |
| Describe workflow | `temporal workflow describe --workflow-id ID` |
| Query workflow | `temporal workflow query --workflow-id ID --type QUERY_NAME` |
| Signal workflow | `temporal workflow signal --workflow-id ID --name SIGNAL --input DATA` |
| Terminate workflow | `temporal workflow terminate --workflow-id ID --reason REASON` |
| List namespaces | `temporal operator namespace list` |
| Search workflows | `temporal workflow list --query 'QUERY'` |


## 🎓 What You Learned

- The **Temporal Web UI** is excellent for browsing executions, reading event history, and visually inspecting running workflows.
- The **`temporal` CLI** is excellent for precise operational actions: list, describe, query, signal, terminate, and batch-manage workflows.
- Queries let you **read** live workflow state, while signals let you **change** it from the outside.
- Batch operations and visibility queries are what turn Temporal from a developer tool into a production operations tool.

### Recap of the 10-notebook journey

Across the full series, you moved from **why workflow engines exist**, to **building workflows and activities**, to **handling failures with sagas and retries**, to **advanced control flow with signals, queries, timers, and testing**, and finally to **production deployment, observability, UI, and CLI operations**. That learning arc mirrors real life: first you write durable workflows, then you learn how to run them safely in the real world.
